In [0]:
%sql
drop table if exists dbcertification.streamschnest.invoices_bz_upd;
drop table if exists dbcertification.streamschnest.customer_rewards_upd;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS dbcertification.streamschnest.customer_rewards_upd (
    CustomerCardNo STRING,
    TotalAmount DOUBLE,
    TotalPoints DOUBLE
);

In [0]:
class Bronze():
    def __init__(self):
        self.base_data_dir = "/Volumes/dbcertification/streamschnest/streamingdata/"
        
    def getSchema(self):
        return """InvoiceNumber string, CreatedTime bigint, StoreID string, PosID string, CashierID string,
                CustomerType string, CustomerCardNo string, TotalAmount double, NumberOfItems bigint, 
                PaymentMethod string, TaxableAmount double, CGST double, SGST double, CESS double, 
                DeliveryType string,
                DeliveryAddress struct<AddressLine string, City string, ContactNumber string, PinCode string, 
                State string>,
                InvoiceLineItems array<struct<ItemCode string, ItemDescription string, 
                    ItemPrice double, ItemQty bigint, TotalValue double>>
            """

    def readInvoices(self):
        from pyspark.sql.functions import col
        return (spark.readStream
                     .format("json")
                     .schema(self.getSchema())
                     .load(f"{self.base_data_dir}/invoices")
                     .withColumn("InputFile", col("_metadata.file_name"))
                )  

    def process(self):
        print(f"\nStarting Bronze Stream...", end='')
        invoicesDF = self.readInvoices()
        sQuery =  ( invoicesDF.writeStream
                            .queryName("bronze-ingestion")
                            .option("checkpointLocation", f"{self.base_data_dir}/checkpoint/invoices_bz_upd")
                            .outputMode("append")
                            .trigger(availableNow=True)
                            .toTable("dbcertification.streamschnest.invoices_bz_upd")           
                    ) 
        print("Done")
        return sQuery   

In [0]:
brnz = Bronze()
query = brnz.process()

In [0]:
%sql
-- first load
select CustomerCardNo, sum(TotalAmount) as TotalAmount ,sum(TotalAmount*0.02) as TotalPoints 
from dbcertification.streamschnest.invoices_bz_upd 
where CustomerCardNo = '2262471989' 
group by CustomerCardNo;

In [0]:
%sql
-- second load
select CustomerCardNo, sum(TotalAmount) as TotalAmount ,sum(TotalAmount*0.02) as TotalPoints 
from dbcertification.streamschnest.invoices_bz_upd 
where CustomerCardNo = '2262471989' 
group by CustomerCardNo;

In [0]:
class Gold():
    def __init__(self):
        self.base_data_dir = "/Volumes/dbcertification/streamschnest/streamingdata/"
        
    def readBronze(self):
        return spark.readStream.table("dbcertification.streamschnest.invoices_bz_upd")

    def getAggregates(self, invoices_df):
        from pyspark.sql.functions import sum, expr
        return (invoices_df.groupBy("CustomerCardNo")
                    .agg(sum("TotalAmount").alias("TotalAmount"),
                         sum(expr("TotalAmount*0.02")).alias("TotalPoints"))
        )

    def upsert(self, rewards_df, batchId):
        rewards_df.createOrReplaceTempView("customers_rewards_df_temp_view")
        merge_statement = """MERGE INTO dbcertification.streamschnest.customer_rewards_upd t
                                USING customers_rewards_df_temp_view s
                                ON s.CustomerCardNo = t.CustomerCardNo
                                WHEN MATCHED THEN
                                UPDATE SET t.TotalAmount =  t.TotalAmount + s.TotalAmount, 
                                            t.TotalPoints = t.TotalPoints + s.TotalPoints
                                WHEN NOT MATCHED
                                THEN INSERT *

        """
        rewards_df.sparkSession.sql(merge_statement)
        # t.TotalAmount + s.TotalAmount,
        # t.TotalPoints + s.TotalPoints
    def saveResults(self, results_df):
        print(f"\nStarting Silver Stream...", end='')
        return (results_df.writeStream
                    .queryName("gold-update")
                    .option("checkpointLocation", f"{self.base_data_dir}/checkpoint/customer_rewards_upd")
                    .outputMode("update")
                    .trigger(availableNow=True) 
                    .foreachBatch(self.upsert)
                    .start() 
                )
        print("Done")

    def process(self):
        invoices_df = self.readBronze()
        aggregate_df = self.getAggregates(invoices_df)
        sQuery = self.saveResults(aggregate_df)
        return sQuery                 

In [0]:
gdStream = Gold()
gdQuery = gdStream.process()

In [0]:
%sql
-- first load
select CustomerCardNo, TotalAmount,TotalPoints from dbcertification.streamschnest.customer_rewards_upd where CustomerCardNo = '2262471989';

In [0]:
%sql
-- second load
select CustomerCardNo, TotalAmount,TotalPoints from dbcertification.streamschnest.customer_rewards_upd where CustomerCardNo = '2262471989';